# Team Season - team_dash_pt_reb

This notebook inspects the data **downloaded** from the `team_dash_pt_reb` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_dash_pt_reb`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [49]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_dash_pt_reb"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [50]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_dash_pt_reb
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_dash_pt_reb
Parquet files: 30


,file,size_mb
0,team_dash_pt_reb__team_id=1610612737.parquet,0.016
1,team_dash_pt_reb__team_id=1610612738.parquet,0.016
2,team_dash_pt_reb__team_id=1610612739.parquet,0.016
3,team_dash_pt_reb__team_id=1610612740.parquet,0.016
4,team_dash_pt_reb__team_id=1610612741.parquet,0.016
5,team_dash_pt_reb__team_id=1610612742.parquet,0.016
6,team_dash_pt_reb__team_id=1610612743.parquet,0.016
7,team_dash_pt_reb__team_id=1610612744.parquet,0.016
8,team_dash_pt_reb__team_id=1610612745.parquet,0.016
9,team_dash_pt_reb__team_id=1610612746.parquet,0.016


In [51]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_dash_pt_reb__team_id=1610612737.parquet,14,25
1,team_dash_pt_reb__team_id=1610612738.parquet,14,25
2,team_dash_pt_reb__team_id=1610612739.parquet,14,25
3,team_dash_pt_reb__team_id=1610612740.parquet,14,25
4,team_dash_pt_reb__team_id=1610612741.parquet,14,25
5,team_dash_pt_reb__team_id=1610612742.parquet,14,25
6,team_dash_pt_reb__team_id=1610612743.parquet,14,25
7,team_dash_pt_reb__team_id=1610612744.parquet,14,25
8,team_dash_pt_reb__team_id=1610612745.parquet,14,25
9,team_dash_pt_reb__team_id=1610612746.parquet,14,25


In [52]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (420, 25)
Total rows: 420
Total columns: 25


In [53]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,420,25,10500,1710,16.286


In [54]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
OVERALL,390,92.857
SHOT_TYPE_RANGE,360,85.714
REB_NUM_CONTESTING_RANGE,330,78.571
REB_DIST_RANGE,300,71.429
SHOT_DIST_RANGE,300,71.429
SORT_ORDER,30,7.143
TEAM_ID,0,0.000
UC_DREB,0,0.000
SEASON_TYPE,0,0.000
SEASON,0,0.000


In [55]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",0,0.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",420,100.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [56]:
# Merge all teams, split into parquets by TABLE_NAME, drop fully-null columns, report nulls, and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    table_col = "TABLE_NAME" if "TABLE_NAME" in df_all.columns else None
    if table_col is None:
        print("Required column not found: TABLE_NAME")
        print(df_all.columns)
    else:
        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)

        table_values = [
            "OverallRebounding",
            "ShotTypeRebounding",
            "NumContestedRebounding",
            "RebDistanceRebounding",
            "ShotDistanceRebounding",
        ]

        null_summary = []
        for name in table_values:
            df_part = df_all.loc[df_all[table_col] == name].copy()

            if df_part.empty:
                print(f"\n{name}: no rows")
                fully_null = []
            else:
                col_nulls = df_part.isna().sum()
                col_null_pct = (df_part.isna().mean() * 100).round(3)
                cols_with_nulls = col_nulls[col_nulls > 0].index.tolist()
                fully_null = col_nulls[col_nulls == len(df_part)].index.tolist()

                print(f"\n{name}: columns with any nulls ({len(cols_with_nulls)})")
                if cols_with_nulls:
                    display(pd.DataFrame({
                        "null_cells": col_nulls[cols_with_nulls],
                        "null_pct": col_null_pct[cols_with_nulls],
                    }).sort_values("null_pct", ascending=False))

                print(f"{name}: fully null columns ({len(fully_null)})")
                if fully_null:
                    print(fully_null)

            # Drop fully-null columns (they belong to other tables)
            if fully_null:
                df_part = df_part.drop(columns=fully_null)

            # Recompute null summary AFTER dropping fully-null columns
            rows, cols = df_part.shape
            total_cells = rows * cols
            null_cells = int(df_part.isna().sum().sum()) if total_cells else 0
            null_pct = (null_cells / total_cells * 100) if total_cells else 0

            null_summary.append({
                "table_name": name,
                "rows": rows,
                "cols": cols,
                "total_cells": total_cells,
                "null_cells": null_cells,
                "null_pct": round(null_pct, 3),
            })

            out_path = out_dir / f"team_dash_pt_reb__{name}.parquet"
            df_part.to_parquet(out_path, index=False)
            print(f"Saved {name}: {len(df_part)} rows -> {out_path}")

        null_summary_df = pd.DataFrame(null_summary).sort_values("table_name")
        print("\nNull summary by TABLE_NAME (after dropping fully-null columns):")
        print(null_summary_df)

        # Also report unexpected table names
        unexpected = sorted(set(df_all[table_col].dropna().unique()) - set(table_values))
        if unexpected:
            print(f"Unexpected TABLE_NAME values: {unexpected}")



OverallRebounding: columns with any nulls (5)


,null_cells,null_pct
SORT_ORDER,30,100.0
SHOT_TYPE_RANGE,30,100.0
REB_NUM_CONTESTING_RANGE,30,100.0
SHOT_DIST_RANGE,30,100.0
REB_DIST_RANGE,30,100.0


OverallRebounding: fully null columns (5)
['SORT_ORDER', 'SHOT_TYPE_RANGE', 'REB_NUM_CONTESTING_RANGE', 'SHOT_DIST_RANGE', 'REB_DIST_RANGE']
Saved OverallRebounding: 30 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_reb__OverallRebounding.parquet

ShotTypeRebounding: columns with any nulls (4)


,null_cells,null_pct
OVERALL,60,100.0
REB_NUM_CONTESTING_RANGE,60,100.0
SHOT_DIST_RANGE,60,100.0
REB_DIST_RANGE,60,100.0


ShotTypeRebounding: fully null columns (4)
['OVERALL', 'REB_NUM_CONTESTING_RANGE', 'SHOT_DIST_RANGE', 'REB_DIST_RANGE']
Saved ShotTypeRebounding: 60 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_reb__ShotTypeRebounding.parquet

NumContestedRebounding: columns with any nulls (4)


...

NumContestedRebounding: fully null columns (4)
['OVERALL', 'SHOT_TYPE_RANGE', 'SHOT_DIST_RANGE', 'REB_DIST_RANGE']
Saved NumContestedRebounding: 90 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_reb__NumContestedRebounding.parquet

RebDistanceRebounding: columns with any nulls (4)


,null_cells,null_pct
OVERALL,120,100.0
SHOT_TYPE_RANGE,120,100.0
REB_NUM_CONTESTING_RANGE,120,100.0
SHOT_DIST_RANGE,120,100.0


RebDistanceRebounding: fully null columns (4)
['OVERALL', 'SHOT_TYPE_RANGE', 'REB_NUM_CONTESTING_RANGE', 'SHOT_DIST_RANGE']
Saved RebDistanceRebounding: 120 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_reb__RebDistanceRebounding.parquet

ShotDistanceRebounding: columns with any nulls (4)


,null_cells,null_pct
OVERALL,120,100.0
SHOT_TYPE_RANGE,120,100.0
REB_NUM_CONTESTING_RANGE,120,100.0
REB_DIST_RANGE,120,100.0


ShotDistanceRebounding: fully null columns (4)
['OVERALL', 'SHOT_TYPE_RANGE', 'REB_NUM_CONTESTING_RANGE', 'REB_DIST_RANGE']
Saved ShotDistanceRebounding: 120 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_reb__ShotDistanceRebounding.parquet

Null summary by TABLE_NAME (after dropping fully-null columns):
               table_name  rows  cols  total_cells  null_cells  null_pct
2  NumContestedRebounding    90    21         1890           0       0.0
0       OverallRebounding    30    20          600           0       0.0
3   RebDistanceRebounding   120    21         2520           0       0.0
4  ShotDistanceRebounding   120    21         2520           0       0.0
1      ShotTypeRebounding    60    21         1260           0       0.0
